# Laboratorio 03 â€” Funciones Avanzadas libres con PySpark

**Semana:** 02 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica las funciones de fecha, agregaciones avanzadas y window functions aprendidas en la Actividad 03 sobre un dataset **de tu elecciÃ³n que tenga al menos una columna de fecha o timestamp**. Puedes reutilizar el dataset del Lab 01 o Lab 02 si ya tiene columna temporal.

Entrega esperada: este notebook completo con todas las celdas ejecutadas y las celdas markdown respondidas.

## Parte 1 â€” DescripciÃ³n del dataset

Documenta tu dataset antes de escribir cÃ³digo:

1. **Nombre y fuente:** Â¿CÃ³mo se llama el dataset y de dÃ³nde lo obtuviste? (incluye URL)
2. **Dominio:** Â¿QuÃ© problema o Ã¡rea describe?
3. **Â¿Por quÃ© lo elegiste?** Â¿QuÃ© pregunta temporal quieres responder?
4. **Columna temporal:** Â¿QuÃ© columna de fecha/timestamp tiene? Â¿QuÃ© granularidad (segundos, minutos, dÃ­as...)?
5. **Columna numÃ©rica principal:** Â¿CuÃ¡l es la mÃ©trica clave del dataset (ventas, duraciÃ³n, precio, puntuaciÃ³n...)?
6. **Columna categÃ³rica para particionar:** Â¿QuÃ© columna usarÃ¡s en el `PARTITION BY` de las window functions?
7. **Preguntas de negocio temporales:** Lista al menos 3 preguntas que solo se puedan responder con funciones de fecha o window functions.

> Requisitp mÃ­nimo del dataset: columna de fecha + columna numÃ©rica + columna categÃ³rica, mÃ¡s de 5.000 filas.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Carga y perfil tÃ©cnico

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col, when, isnan, sum as spark_sum

VOL = "/Volumes/workspace/default/week_2"  # ajusta si usas un volumen distinto
ARCHIVO = "tu_archivo.csv"                 # cambia por el nombre real

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

print(f"Filas: {df_raw.count():,} | Columnas: {len(df_raw.columns)}")
df_raw.printSchema()

In [ ]:
df_raw.describe().show(truncate=False)

In [ ]:
# Nulos y vacÃ­os
total = df_raw.count()
numeric_types = {"double", "float", "long", "integer", "short", "byte"}
nulos = df_raw.select([
    spark_sum(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df_raw.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
            (col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
]).collect()[0].asDict()

print(f"{'Columna':<35} {'Nulos':>8} {'%':>8}")
print("-" * 54)
for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    print(f"{c:<35} {n:>8,} {n/total*100:>7.1f}%")

**Observaciones del perfil:** Â¿Hay nulos en la columna temporal o numÃ©rica? Â¿QuÃ© harÃ¡s con ellos antes de los anÃ¡lisis?

## Parte 3 â€” PreparaciÃ³n de la columna temporal

AsegÃºrate de que la columna de fecha estÃ© en tipo `timestamp` o `date`. Luego extrae al menos **4 componentes temporales** que tengan sentido para tu dataset.

In [ ]:
# Ajusta el nombre de la columna temporal y el formato si es necesario
COL_FECHA = "fecha"        # cambia por el nombre real
COL_NUMERICA = "valor"     # cambia por la mÃ©trica clave
COL_CATEGORIA = "categoria" # cambia por la columna categÃ³rica

df = df_raw \
    .withColumn("ts", F.to_timestamp(COL_FECHA)) \
    .withColumn("hora",        F.hour("ts")) \
    .withColumn("dia_semana",  F.dayofweek("ts")) \
    .withColumn("mes",         F.month("ts")) \
    .withColumn("anio",        F.year("ts")) \
    .withColumn("semana_anio", F.weekofyear("ts")) \
    .withColumn("mes_inicio",  F.date_trunc("month", "ts"))

df.select("ts", "hora", "dia_semana", "mes", "anio", "semana_anio").show(10)

**Â¿A quÃ© granularidad tienen mÃ¡s sentido los anÃ¡lisis con este dataset?** (hora, dÃ­a, semana, mes) Â¿Por quÃ©?

## Parte 4 â€” Agregaciones avanzadas por perÃ­odo

Calcula al menos **3 mÃ©tricas distintas** agrupando por perÃ­odo temporal. Muestra la evoluciÃ³n y comenta si ves tendencias, estacionalidad o anomalÃ­as.

In [ ]:
# AgrupaciÃ³n por mes â€” ajusta COL_NUMERICA segÃºn tu dataset
df.groupBy("mes_inicio") \
    .agg(
        F.count("*").alias("registros"),
        F.sum(COL_NUMERICA).alias("total"),
        F.avg(COL_NUMERICA).alias("promedio"),
        F.percentile_approx(COL_NUMERICA, 0.9).alias("percentil_90")
    ) \
    .orderBy("mes_inicio") \
    .show(24, truncate=False)

**Â¿QuÃ© tendencias, picos o anomalÃ­as ves en la evoluciÃ³n temporal?**

In [ ]:
# Agrega una segunda agrupaciÃ³n por otro componente temporal (dÃ­a de semana, hora, etc.)
# que tenga sentido para tu dominio

**Observaciones:**

## Parte 5 â€” Window Function: Ranking

Usa `rank()` o `dense_rank()` para clasificar registros dentro de grupos. Define un `PARTITION BY` y `ORDER BY` que tenga sentido para tu dominio.

In [ ]:
# Ajusta la particiÃ³n y el orden segÃºn tu dataset
windowSpec_rank = Window.partitionBy(COL_CATEGORIA).orderBy(F.col(COL_NUMERICA).desc())

df_ranking = df.withColumn("rank_en_grupo", F.rank().over(windowSpec_rank))

# Muestra los top 3 por grupo
df_ranking.filter(F.col("rank_en_grupo") <= 3) \
    .orderBy(COL_CATEGORIA, "rank_en_grupo") \
    .select(COL_CATEGORIA, COL_NUMERICA, "rank_en_grupo") \
    .show(30, truncate=False)

**Explica:** Â¿QuÃ© `PARTITION BY` y `ORDER BY` elegiste y por quÃ© tiene sentido en tu dominio? Â¿QuÃ© diferencia hay entre `rank()` y `dense_rank()` en tu resultado?

## Parte 6 â€” Window Function: LAG y detecciÃ³n de variaciones

Calcula la variaciÃ³n respecto a la observaciÃ³n anterior dentro de cada particiÃ³n. Identifica los casos donde la variaciÃ³n es mÃ¡s pronunciada.

In [ ]:
windowSpec_lag = Window.partitionBy(COL_CATEGORIA).orderBy("ts")

df = df \
    .withColumn("valor_anterior", F.lag(COL_NUMERICA, 1).over(windowSpec_lag)) \
    .withColumn(
        "variacion",
        F.round(F.col(COL_NUMERICA) - F.col("valor_anterior"), 2)
    )

# Top 10 variaciones mÃ¡s grandes
df.filter(F.col("variacion").isNotNull()) \
    .orderBy(F.col("variacion").desc()) \
    .select(COL_CATEGORIA, "ts", COL_NUMERICA, "valor_anterior", "variacion") \
    .show(10, truncate=False)

**Â¿QuÃ© representan las variaciones mÃ¡s grandes? Â¿Son anomalÃ­as, eventos esperados, o artefactos del dataset?**

## Parte 7 â€” Window Function: Acumulado o Media mÃ³vil

Implementa **una** de las dos opciones (la que tenga mÃ¡s sentido para tu dataset):

- **OpciÃ³n A â€” Acumulado:** total acumulado de la mÃ©trica principal, ordenado por fecha, particionado por categorÃ­a
- **OpciÃ³n B â€” Media mÃ³vil:** media de los Ãºltimos N registros (elige N con criterio de dominio)

Documenta cuÃ¡l elegiste y por quÃ©.

In [ ]:
# OpciÃ³n A â€” Acumulado
# windowSpec_acum = Window.partitionBy(COL_CATEGORIA) \
#     .orderBy("ts") \
#     .rowsBetween(Window.unboundedPreceding, Window.currentRow)
# df = df.withColumn("acumulado", F.sum(COL_NUMERICA).over(windowSpec_acum))

# OpciÃ³n B â€” Media mÃ³vil (Ãºltimos N registros)
# N = 7
# windowSpec_rolling = Window.partitionBy(COL_CATEGORIA) \
#     .orderBy("ts") \
#     .rowsBetween(-N + 1, 0)
# df = df.withColumn("media_movil", F.round(F.avg(COL_NUMERICA).over(windowSpec_rolling), 2))

# Descomenta la opciÃ³n que elijas y ajusta los parÃ¡metros

**Â¿CuÃ¡l opciÃ³n elegiste y por quÃ© tiene sentido para tu dataset? Si elegiste media mÃ³vil, Â¿por quÃ© ese valor de N?**

## Parte 8 â€” AnÃ¡lisis de negocio: 3 preguntas temporales

Responde las 3 preguntas temporales que planteaste en la Parte 1. Cada respuesta debe usar al menos una funciÃ³n de fecha o una window function. Incluye:
- Bloque de cÃ³digo PySpark
- Celda markdown con la conclusiÃ³n en lenguaje natural

In [ ]:
# Pregunta 1:

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2:

**ConclusiÃ³n pregunta 2:**

In [ ]:
# Pregunta 3:

**ConclusiÃ³n pregunta 3:**

## Parte 9 â€” ReflexiÃ³n final

Responde en esta celda:

1. Â¿CuÃ¡l window function te resultÃ³ mÃ¡s difÃ­cil de entender o aplicar? Â¿Por quÃ©?
2. Â¿En quÃ© se diferencia una window function de un `groupBy` para responder la misma pregunta?
3. Â¿QuÃ© hallazgo temporal te sorprendiÃ³ mÃ¡s en tu dataset?
4. Â¿QuÃ© anÃ¡lisis adicional harÃ­as si tuvieras mÃ¡s columnas o mÃ¡s historia temporal?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_02/laboratorios/lab_03_funciones_avanzadas.ipynb semana_02/laboratorios/<tu-nombre>/lab_03_funciones_avanzadas.ipynb

git add semana_02/laboratorios/<tu-nombre>/lab_03_funciones_avanzadas.ipynb
git commit -m "lab: semana02 lab03 window functions <nombre-dataset> - <tu-nombre>"
git push origin develop
```